In [ ]:
import torch
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
import os

In [ ]:
def compare_tensors_1based(t1, t2):
    # 1. 크기 다르면 스킵
    if t1.shape != t2.shape:
        print("Pass (Size mismatch)")
        return

    # 2. 틀린 위치 찾기 (Tensor 형태)
    diff_indices = torch.nonzero(t1 != t2).flatten()
    
    if len(diff_indices) > 0:
        # [핵심] 텐서에 1을 더해서 1-based index로 변환
        diff_1based = diff_indices + 1 
        
        # 리스트로 변환 후 쉼표로 출력
        print(*diff_1based.tolist(), sep='+')
    else:
        print("All Match")

In [ ]:
CSV_PATH = "/data/esmdiff_target/EigenFold/splits/apo.csv"
DIR_GT_TOKEN = "/data/esmdiff_target/EigenFold/structures/tokenize"

apo_tokens = []
holo_tokens = []
apo_name = []
holo_name = []
df = pd.read_csv(CSV_PATH)
for _, row in df.iterrows():
    p_name = row['name']
    apo_name.append(p_name)
    h_name = row['holo']
    holo_name.append(h_name)
    entry = {"name": p_name}
    GT_Apo = os.path.join(DIR_GT_TOKEN, f"{p_name[:-4]}.ptm")
    temp_apo_token = torch.load(GT_Apo, map_location=torch.device('cpu'))
    apo_tokens.append( temp_apo_token['small_codebook'][1:-1] )
    
    GT_Holo = os.path.join(DIR_GT_TOKEN, f"{h_name[:-4]}.ptm")
    temp_holo_token = torch.load(GT_Holo, map_location=torch.device('cpu'))
    holo_tokens.append( temp_holo_token['small_codebook'][1:-1] )

In [ ]:
for i in range(len(apo_tokens)):
    print(f"color red, {apo_name[i][:-4]} and resi", end=' ')
    compare_tensors_1based(apo_tokens[i], holo_tokens[i])
    print(f"color red, {holo_name[i][:-4]} and resi", end=' ')
    compare_tensors_1based(apo_tokens[i], holo_tokens[i])